<a href="https://colab.research.google.com/github/khovan123/cropstate/blob/main/notebooks/cropstate_colab_github_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CROPSTATE Colab Workflow

- Code clone từ GitHub; Dataset/KB/Results đọc-ghi trên Google Drive (`MyDrive/CROPSTATE_DATASET`, `CROPSTATE_KNOWLEDGE_BASE`, `CROPSTATE_RESULTS`).
- Dataset = **RiceSEG** (segmentation masks). Nhãn 6 stage suy từ mask, không có sẵn.
- Trước khi train: `Runtime > Change runtime type` → chọn **GPU**.

## Thứ tự chạy

**A. Chuẩn bị**
1. Mount Drive → 2. Clone/pull repo → 3. Cài deps → 4. Kiểm tra dataset/KB.

**B. Vision (RiceSEG, leak-free)**
5. **Cell 5** — gán nhãn 6 stage BBCH từ RiceSEG → `RICESEG_MANIFEST` (đọc `class_pixel_counts.xlsx`, suy từ mask) + audit manifest.
6. **Train `vision_final`** — dùng `--manifest {RICESEG_MANIFEST}`; split leak-free + balanced sampler tự áp. Rồi fine-tune, xem output, predict ảnh upload.
7. **Novelty** — retrain ordinal/focal + fixed-split multi-seed, bảng so sánh, phân tích post-hoc (calibration/temporal/leakage).

**C. Retrieval / Knowledge base** (độc lập với vision)
8. Audit corpus IRRI → run/evaluate retrieval (2 kịch bản minh hoạ) → retrieval đóng vòng từ belief thật (robust).

> ⚠️ Chuỗi layout cũ (folder `01_..06_`, encode parent, build/convert image manifest) **đã gỡ** — dataset giờ là RiceSEG, nhãn stage do cell 5 sinh. "Run all" an toàn.


In [6]:
# Sửa REPO_URL thành GitHub repo thật của bạn.
REPO_URL = "https://github.com/khovan123/cropstate"
PROJECT_DIR = "/content/CROPSTATE_Full_Research_Package"

DATA_ROOT = "/content/drive/MyDrive/CROPSTATE_DATASET"
KNOWLEDGE_ROOT = "/content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE"
RESULTS_ROOT = "/content/drive/MyDrive/CROPSTATE_RESULTS"
OUTPUT_DIR = f"{RESULTS_ROOT}/vision_final"

# Knowledge base: CHỈ dùng corpus IRRI (build từ raw_sources_irri/). Không dùng
# raw_sources hỗn hợp (PDF tiếng Việt/English) hay chunking cũ nữa.
KB_COMPLETE_CORPUS = f"{KNOWLEDGE_ROOT}/chunks/rice_knowledge_irri_en.jsonl"
KB_NONRESTRICTED_CORPUS = f"{KNOWLEDGE_ROOT}/chunks/rice_knowledge_irri_en_nonrestricted.jsonl"
RETRIEVAL_SCENARIOS = "data/retrieval_scenarios.csv"
# Dataset = RiceSEG (segmentation masks) -> gán nhãn 6 stage vào manifest này.
RICESEG_MANIFEST = "data/riceseg_stage_manifest.csv"

print("REPO_URL:", REPO_URL)
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("KNOWLEDGE_ROOT:", KNOWLEDGE_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("RICESEG_MANIFEST:", RICESEG_MANIFEST)

REPO_URL: https://github.com/khovan123/cropstate
PROJECT_DIR: /content/CROPSTATE_Full_Research_Package
DATA_ROOT: /content/drive/MyDrive/CROPSTATE_DATASET
KNOWLEDGE_ROOT: /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE
RESULTS_ROOT: /content/drive/MyDrive/CROPSTATE_RESULTS
RICESEG_MANIFEST: data/riceseg_stage_manifest.csv


## 1. Mount Google Drive

In [7]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4b. Preflight — tạo & kiểm tra thư mục kết quả trên Drive

Lỗi `FileNotFoundError` khi ghi `CROPSTATE_RESULTS/...` (cell #12, #13, #19, #20) do Drive FUSE:
`mkdir` báo OK nhưng thư mục chưa đồng bộ kịp lúc ghi, hoặc `CROPSTATE_RESULTS` chưa tồn tại trong MyDrive.
Cell này tạo sẵn mọi thư mục **sớm** và probe ghi/đọc (có retry) trước khi chạy các bước nặng.

In [8]:
# Preflight: tạo & kiểm tra ghi được vào CROPSTATE_RESULTS trên Drive.
import os, time
from pathlib import Path

SUBDIRS = ["", "retrieval", "vision_final", "novelty", "novelty/logits", "novelty/logits_sig"]
for s in SUBDIRS:
    os.makedirs(os.path.join(RESULTS_ROOT, s), exist_ok=True)

def _probe(dir_path, retries=6, delay=2):
    p = Path(dir_path) / ".write_probe"
    for _ in range(retries):
        try:
            p.write_text("ok")
            assert p.read_text() == "ok"
            p.unlink()
            return True
        except (FileNotFoundError, OSError):
            time.sleep(delay)  # chờ Drive FUSE đồng bộ
    return False

bad = [s or "(root)" for s in SUBDIRS if not _probe(os.path.join(RESULTS_ROOT, s))]
if bad:
    raise RuntimeError(
        f"Không ghi được vào {RESULTS_ROOT}: {bad}.\n"
        "Khắc phục: tạo thủ công folder 'CROPSTATE_RESULTS' ngay trong MyDrive "
        "(không để là shortcut tới Drive chia sẻ chỉ-đọc), Runtime > Restart, chạy lại từ cell mount."
    )
print("Preflight OK — ghi được vào", RESULTS_ROOT)
for s in SUBDIRS:
    print("  ", os.path.join(RESULTS_ROOT, s or ""))

Preflight OK — ghi được vào /content/drive/MyDrive/CROPSTATE_RESULTS
   /content/drive/MyDrive/CROPSTATE_RESULTS/
   /content/drive/MyDrive/CROPSTATE_RESULTS/retrieval
   /content/drive/MyDrive/CROPSTATE_RESULTS/vision_final
   /content/drive/MyDrive/CROPSTATE_RESULTS/novelty
   /content/drive/MyDrive/CROPSTATE_RESULTS/novelty/logits
   /content/drive/MyDrive/CROPSTATE_RESULTS/novelty/logits_sig


## 2. Clone hoặc update repo từ GitHub

In [9]:
import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
print("Current directory:", os.getcwd())

Current directory: /content/CROPSTATE_Full_Research_Package


## 3. Cài dependencies

In [10]:
!pip install -r requirements.txt
!pip install -e .

Obtaining file:///content/CROPSTATE_Full_Research_Package
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cropstate-research (pyproject.toml) ... done
  Created wheel for cropstate-research: filename=cropstate_research-0.1.0-0.editable-py3-none-any.whl size=1383 sha256=465a06c11743feea69960624eed8764f72414175e6189ec7f984f6953c75b3b3
  Stored in directory: /tmp/pip-ephem-wheel-cache-300c7qsa/wheels/37/37/8f/89550734e79d9e7fc68e041f688e3bc8f62ff3b93e9d2c762c
Successfully built cropstate-research
  Attempting uninstall: cropstate-research
    Found existing installation: cropstate-research 0.1.0
    Uninstalling cropstate-research-0.1.0:
      Successfully uninstalled cropstate-research-0.1.0


## 4. Kiểm tra dataset và knowledge base trên Drive

In [11]:
!ls -lah "{DATA_ROOT}"
!ls -lah "{KNOWLEDGE_ROOT}"

lrw------- 1 root root 0 Jul 11 17:03 /content/drive/MyDrive/CROPSTATE_DATASET -> /content/drive/.shortcut-targets-by-id/1iazxjNfESn9lm7fn-1L8ETOAtKSWVDYN/CROPSTATE_DATASET
lrw------- 1 root root 0 Jul 11 17:04 /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE -> /content/drive/.shortcut-targets-by-id/1rwtO-iHapZuwjFhRqwYcXm9RTyrvuHqS/CROPSTATE_KNOWLEDGE_BASE


Dataset trên Drive giờ là **RiceSEG** (không còn folder `01_..06_`):

```text
CROPSTATE_DATASET/
  class_pixel_counts.xlsx      # pixel 6 lớp / mask (bg,green,senescent,panicle,weeds,duckweed)
  crop_mapping.xlsx
  segmentation/<nước>/<vùng>/label/*.png   # mask index 0-5
                          /rgb/*.jpg     # ảnh RGB tương ứng
```

Nhãn 6 stage được **suy từ mask** ở cell kế (không có sẵn trong dataset).

Knowledge base **chỉ dùng corpus IRRI**: `chunks/rice_knowledge_irri_en.jsonl` + `..._nonrestricted.jsonl` + `raw_sources_irri/`.


## 5. Gán nhãn 6 stage BBCH từ RiceSEG

Đọc `class_pixel_counts.xlsx`, suy giai đoạn từ hình thái trong mask (**bông** = panicle, **độ úa** = senescent, **độ phủ** = cover) → manifest 6 lớp `RICESEG_MANIFEST`, tương thích thẳng `train_vision.py` (leak-free split + balanced sampler tự áp).

- 3 lớp muộn (reproductive/grain_filling/ripening): dựa panicle+senescence → **chắc**.
- 3 lớp sớm (establishment/tillering/stem_booting): dựa cover → **yếu**; cột `review_flag` đánh dấu ca sát ranh giới. Chỉnh `--cover-low/-high`, `--senescent-low/-high` nếu lệch.


In [12]:
# RiceSEG -> manifest 6 stage (image_path tương đối DATA_ROOT, split=unassigned).
!PYTHONPATH=src python scripts/experiments/relabel_riceseg_bbch.py \
  --dataset-root "{DATA_ROOT}" \
  --output {RICESEG_MANIFEST}

# Kiểm tra manifest (cột, phân bố, leakage, file thiếu)
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest {RICESEG_MANIFEST} --data-root "{DATA_ROOT}"

RiceSEG rows: 3078 | unresolved rgb: 100 | dropped near-empty: 78 | labelled: 2900
Wrote data/riceseg_stage_manifest.csv

Stage distribution (review-flagged in parentheses):
  establishment    727  (208 to review)
  tillering        852  (417 to review)
  stem_booting     794  (194 to review)
  reproductive     288  (222 to review)
  grain_filling    110  (84 to review)
  ripening         129  (9 to review)
  parents (leak-free groups): 742
  flagged for review: 1134/2900
Rows: 2900

Class counts:
 macro_stage
tillering        852
stem_booting     794
establishment    727
reproductive     288
ripening         129
grain_filling    110
Name: count, dtype: int64

Split counts:
 split
unassigned    2900
Name: count, dtype: int64

parent_image_id leakage groups: 0

field_id leakage groups: 0

capture_session leakage groups: 0

Unlabeled rows: 0

Missing image files: 0

Non-training labels in manifest: 0


## 9. Audit corpus IRRI (complete)

Audit corpus IRRI đầy đủ (`rice_knowledge_irri_en.jsonl`) để xem coverage, restricted chunks và trạng thái production. Output lưu vào Drive results.

In [13]:
!mkdir -p "{RESULTS_ROOT}/retrieval"
!PYTHONPATH=src python scripts/audit_knowledge_base.py \
  --input "{KB_COMPLETE_CORPUS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/knowledge_audit_complete.json"


{
  "total_chunks": 173,
  "by_topic": {
    "disease_risk": 11,
    "general_crop_care": 16,
    "harvest_readiness": 18,
    "nutrient_management": 8,
    "pest_risk": 60,
    "residue_management": 16,
    "water_management": 16,
    "weed_management": 28
  },
  "by_facet": {
    "conditions": 48,
    "fertilizer": 10,
    "general": 73,
    "next_stage_action": 7,
    "pest_disease_prevention": 35
  },
  "by_source": {
    "IRRI_RKB_001": 8,
    "IRRI_RKB_002": 12,
    "IRRI_RKB_003": 5,
    "IRRI_RKB_004": 18,
    "IRRI_RKB_005": 8,
    "IRRI_RKB_006": 6,
    "IRRI_RKB_007": 25,
    "IRRI_RKB_008": 22,
    "IRRI_RKB_009": 17,
    "IRRI_RKB_010": 13,
    "IRRI_RKB_011": 9,
    "IRRI_RKB_012": 10,
    "IRRI_RKB_013": 11,
    "IRRI_RKB_014": 9
  },
  "by_review_status": {
    "machine_curated_pending_domain_review": 173
  },
  "stage_high_compatibility": {
    "establishment": 83,
    "tillering": 4,
    "stem_booting": 2,
    "reproductive": 5,
    "grain_filling": 1,
    "ripening":

## 10. Audit corpus IRRI (non-restricted)

Corpus IRRI đã loại chunk có restricted action (`rice_knowledge_irri_en_nonrestricted.jsonl`), phù hợp cho retrieval pilot.

In [14]:
!PYTHONPATH=src python scripts/audit_knowledge_base.py \
  --input "{KB_NONRESTRICTED_CORPUS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/knowledge_audit_nonrestricted.json"


{
  "total_chunks": 173,
  "by_topic": {
    "disease_risk": 11,
    "general_crop_care": 16,
    "harvest_readiness": 18,
    "nutrient_management": 8,
    "pest_risk": 60,
    "residue_management": 16,
    "water_management": 16,
    "weed_management": 28
  },
  "by_facet": {
    "conditions": 48,
    "fertilizer": 10,
    "general": 73,
    "next_stage_action": 7,
    "pest_disease_prevention": 35
  },
  "by_source": {
    "IRRI_RKB_001": 8,
    "IRRI_RKB_002": 12,
    "IRRI_RKB_003": 5,
    "IRRI_RKB_004": 18,
    "IRRI_RKB_005": 8,
    "IRRI_RKB_006": 6,
    "IRRI_RKB_007": 25,
    "IRRI_RKB_008": 22,
    "IRRI_RKB_009": 17,
    "IRRI_RKB_010": 13,
    "IRRI_RKB_011": 9,
    "IRRI_RKB_012": 10,
    "IRRI_RKB_013": 11,
    "IRRI_RKB_014": 9
  },
  "by_review_status": {
    "machine_curated_pending_domain_review": 173
  },
  "stage_high_compatibility": {
    "establishment": 83,
    "tillering": 4,
    "stem_booting": 2,
    "reproductive": 5,
    "grain_filling": 1,
    "ripening":

## 12. Run retrieval mẫu

Ví dụ: water management cho stage tillering. Script sẽ tải multilingual embedding model lần đầu chạy.

In [15]:
!PYTHONPATH=src python scripts/run_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --topic water_management \
  --stage tillering \
  --mode research \
  --top-k 5 \
  --output "{RESULTS_ROOT}/retrieval/water_tillering.json"


modules.json: 100% 229/229 [00:00<00:00, 499kB/s]
config_sentence_transformers.json: 100% 122/122 [00:00<00:00, 792kB/s]
README.md: 100% 3.89k/3.89k [00:00<00:00, 17.5MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 277kB/s]
config.json: 100% 645/645 [00:00<00:00, 4.52MB/s]
model.safetensors: 100% 471M/471M [00:04<00:00, 105MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 18959.78it/s]
tokenizer_config.json: 100% 526/526 [00:00<00:00, 2.64MB/s]
tokenizer.json: 100% 9.08M/9.08M [00:00<00:00, 10.5MB/s]
special_tokens_map.json: 100% 239/239 [00:00<00:00, 910kB/s]
config.json: 100% 190/190 [00:00<00:00, 1.10MB/s]
{
  "topic": "water_management",
  "query": "Bằng chứng quản lý nước phù hợp với lúa ở giai đoạn Tillering, BBCH 20-29.",
  "belief": [
    0.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0
  ],
  "confidence": 1.0,
  "mode": "research",
  "results": [
    {
      "rank": 1,
      "chunk_id": "IRRI_RKB_007_C0005",
      "text": "Large amounts of water can be lost duri

## 12b. Liệt kê chunk_id thật để tạo `data/retrieval_scenarios.csv`

Cell "Evaluate retrieval baselines" bên dưới cần file `data/retrieval_scenarios.csv` — đây là file bạn tự viết tay (ground truth đánh giá), không phải file do script tự sinh ra. Nếu file này chưa tồn tại, cell evaluate sẽ báo `FileNotFoundError`, đó là điều bình thường, không phải lỗi.

Cell này chỉ **liệt kê chunk_id thật** trong corpus của bạn theo từng topic/stage, để bạn có cơ sở chọn:

- `relevant_chunk_ids`: các chunk phù hợp giai đoạn đó (stage_compatibility ≥ 0.8);
- `incompatible_chunk_ids`: các chunk cùng topic nhưng không phù hợp giai đoạn đó (stage_compatibility = 0).

**Quan trọng:** danh sách này chỉ là gợi ý tự động dựa trên vector `stage_compatibility` sẵn có trong corpus. Không nên dùng thẳng làm ground truth khoa học cho paper, vì `stage_compatibility` cũng chính là tín hiệu mà bộ rerank dùng để xếp hạng — dùng lại nó làm nhãn đánh giá sẽ gây thiên lệch có lợi cho chính phương pháp đang được đánh giá (circular evaluation). Hãy dùng danh sách này làm điểm khởi đầu, sau đó tự rà soát/chỉnh sửa (hoặc nhờ người có chuyên môn nông nghiệp xác nhận) trước khi đưa vào `data/retrieval_scenarios.csv` thật.


In [16]:
import sys
sys.path.insert(0, "src")

from cropstate.knowledge import load_knowledge_chunks
from cropstate.constants import STAGE_NAMES

chunks = load_knowledge_chunks(KB_NONRESTRICTED_CORPUS, mode="research")
print(f"Total chunks loaded: {len(chunks)}\n")

topics = sorted({c.topic for c in chunks})
for topic in topics:
    topic_chunks = [c for c in chunks if c.topic == topic]
    print(f"=== topic: {topic} ({len(topic_chunks)} chunks) ===")
    for stage_index, stage in enumerate(STAGE_NAMES):
        relevant = [c for c in topic_chunks if c.stage_compatibility[stage_index] >= 0.5]
        incompatible = [c for c in topic_chunks if c.stage_compatibility[stage_index] == 0.0]
        if not relevant and not incompatible:
            continue
        print(f"  [{stage}]")
        if relevant:
            print(f"    relevant_chunk_ids candidates ({len(relevant)}):")
            for c in relevant[:8]:
                print(f"      {c.chunk_id}  -  {c.text[:70]}...")
        if incompatible:
            ids_preview = "|".join(c.chunk_id for c in incompatible[:8])
            print(f"    incompatible_chunk_ids candidates ({len(incompatible)}): {ids_preview}")
    print()

print("Copy các chunk_id phù hợp vào data/retrieval_scenarios.csv, cột relevant_chunk_ids")
print("và incompatible_chunk_ids cách nhau bằng dấu '|', ví dụ: kb_water_01|kb_water_07")


Total chunks loaded: 173

=== topic: disease_risk (11 chunks) ===
  [establishment]
    relevant_chunk_ids candidates (3):
      IRRI_RKB_003_C0005  -  Rice varieties should have Good grain quality (especially cooking char...
      IRRI_RKB_004_C0004  -  It must be grown, harvested, and processed correctly for best yield an...
      IRRI_RKB_004_C0017  -  The amount of moisture should be less than 14%, and preferably less th...
  [tillering]
    relevant_chunk_ids candidates (2):
      IRRI_RKB_003_C0005  -  Rice varieties should have Good grain quality (especially cooking char...
      IRRI_RKB_004_C0004  -  It must be grown, harvested, and processed correctly for best yield an...
    incompatible_chunk_ids candidates (1): IRRI_RKB_004_C0017
  [stem_booting]
    relevant_chunk_ids candidates (1):
      IRRI_RKB_003_C0005  -  Rice varieties should have Good grain quality (especially cooking char...
    incompatible_chunk_ids candidates (2): IRRI_RKB_004_C0004|IRRI_RKB_004_C0017
  [repr

## 13. Evaluate retrieval baselines

Chỉ chạy cell này khi đã có `data/retrieval_scenarios.csv`. Metrics gồm P@k, R@k, nDCG@k, SIRR@k cho ungated, hard top-1, fixed-soft, adaptive-soft và oracle.

In [17]:
!PYTHONPATH=src python scripts/evaluate_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --scenarios "{RETRIEVAL_SCENARIOS}" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/retrieval_evaluation.json"


Loading weights: 100% 199/199 [00:00<00:00, 6197.64it/s]
{
  "B0_ungated": {
    "precision_at_k": 0.1,
    "recall_at_k": 0.1,
    "ndcg_at_k": 0.06560253875617089,
    "sirr_at_k": 0.2
  },
  "B1_query_expansion": {
    "precision_at_k": 0.0,
    "recall_at_k": 0.0,
    "ndcg_at_k": 0.0,
    "sirr_at_k": 0.2
  },
  "B2_hard_filter": {
    "precision_at_k": 0.1,
    "recall_at_k": 0.16666666666666666,
    "ndcg_at_k": 0.11731968150568911,
    "sirr_at_k": 0.1
  },
  "B3_fixed_soft": {
    "precision_at_k": 0.2,
    "recall_at_k": 0.26666666666666666,
    "ndcg_at_k": 0.18292222026186,
    "sirr_at_k": 0.1
  },
  "P_adaptive_soft": {
    "precision_at_k": 0.1,
    "recall_at_k": 0.1,
    "ndcg_at_k": 0.06560253875617089,
    "sirr_at_k": 0.2
  },
  "oracle_reference": {
    "precision_at_k": 0.2,
    "recall_at_k": 0.26666666666666666,
    "ndcg_at_k": 0.30767353793273144,
    "sirr_at_k": 0.0
  }
}


## 19. Train lần đầu vào `vision_final`

Chỉ dùng một output folder duy nhất. Script tự tạo `OUTPUT_DIR/manifest.csv` từ các stage folder trong `DATA_ROOT`, rồi lưu checkpoint và test metrics vào cùng folder.

In [18]:
# Vision final trên RiceSEG. Manifest do cell 5 tạo; split LEAK-FREE + balanced sampler tự áp.
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest {RICESEG_MANIFEST} \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --balanced-sampler \
  --output "{OUTPUT_DIR}"

[train] device=cuda | manifest 2900 rows
[train] hashing 2900 images for leak-free groups (threshold=10)...
[train] split -> {'train': 2400, 'test': 263, 'validation': 237}
model.safetensors: 100% 46.8M/46.8M [00:01<00:00, 44.0MB/s]
[epoch 1] phase=head_only val_acc=0.063 val_f1=0.046 val_loss=2.064
[epoch 2] phase=head_only val_acc=0.076 val_f1=0.064 val_loss=2.047
[epoch 3] phase=head_only val_acc=0.089 val_f1=0.095 val_loss=1.908
[epoch 4] phase=full_finetune val_acc=0.097 val_f1=0.105 val_loss=1.744
[epoch 5] phase=full_finetune val_acc=0.160 val_f1=0.205 val_loss=1.557
[epoch 6] phase=full_finetune val_acc=0.321 val_f1=0.353 val_loss=1.353
[epoch 7] phase=full_finetune val_acc=0.460 val_f1=0.469 val_loss=1.224
[epoch 8] phase=full_finetune val_acc=0.574 val_f1=0.573 val_loss=1.116
[epoch 9] phase=full_finetune val_acc=0.624 val_f1=0.619 val_loss=1.041
[epoch 10] phase=full_finetune val_acc=0.696 val_f1=0.674 val_loss=0.963
[epoch 11] phase=full_finetune val_acc=0.700 val_f1=0.687 

## 20. Fine-tune tiếp trong cùng `vision_final`

Dùng cell này cho lần chạy tiếp theo. Checkpoint cũ được đọc từ `OUTPUT_DIR/best_checkpoint.pt`; nếu validation tốt hơn, checkpoint mới ghi đè vào đúng file đó.

In [19]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{OUTPUT_DIR}/manifest.csv" \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --resume-checkpoint "{OUTPUT_DIR}/best_checkpoint.pt" \
  --freeze-backbone-epochs 0 \
  --learning-rate 0.0001 \
  --backbone-learning-rate 0.00001 \
  --output "{OUTPUT_DIR}"

[train] device=cuda | manifest 2900 rows
[epoch 31] phase=full_finetune val_acc=0.789 val_f1=0.758 val_loss=0.636
[epoch 32] phase=full_finetune val_acc=0.781 val_f1=0.761 val_loss=0.604
[epoch 33] phase=full_finetune val_acc=0.797 val_f1=0.782 val_loss=0.580
[epoch 34] phase=full_finetune val_acc=0.797 val_f1=0.781 val_loss=0.578
[epoch 35] phase=full_finetune val_acc=0.785 val_f1=0.758 val_loss=0.579
[epoch 36] phase=full_finetune val_acc=0.789 val_f1=0.766 val_loss=0.605
[epoch 37] phase=full_finetune val_acc=0.793 val_f1=0.763 val_loss=0.586
[epoch 38] phase=full_finetune val_acc=0.810 val_f1=0.793 val_loss=0.581
[epoch 39] phase=full_finetune val_acc=0.785 val_f1=0.779 val_loss=0.582
[epoch 40] phase=full_finetune val_acc=0.814 val_f1=0.801 val_loss=0.576
[epoch 41] phase=full_finetune val_acc=0.810 val_f1=0.797 val_loss=0.596
[epoch 42] phase=full_finetune val_acc=0.797 val_f1=0.787 val_loss=0.586
[epoch 43] phase=full_finetune val_acc=0.797 val_f1=0.794 val_loss=0.582
[epoch 44]

## 21. Xem output đã lưu trên Drive

Folder này chỉ nên có `best_checkpoint.pt`, `history.json`, `class_counts.json`, `manifest.csv`, và `test_metrics.json`.

In [20]:
!ls -lah "{OUTPUT_DIR}"

total 44M
-rw------- 1 root root  43M Jul 11 19:54 best_checkpoint.pt
-rw------- 1 root root  477 Jul 11 19:59 class_counts.json
-rw------- 1 root root 139K Jul 11 19:59 history.json
-rw------- 1 root root 609K Jul 11 19:43 manifest.csv
-rw------- 1 root root 2.0K Jul 11 19:59 test_metrics.json


## 22. Test một ảnh upload từ máy

In [21]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
print("Uploaded:", image_path)

Saving cay-lua.png to cay-lua (1).png
Uploaded: cay-lua (1).png


In [ ]:
!PYTHONPATH=src python scripts/predict_image.py \
  --checkpoint "{OUTPUT_DIR}/best_checkpoint.pt" \
  --image "{image_path}"

{
  "image_path": "cay-lua (1).png",
  "predicted_id": 5,
  "predicted_stage": "ripening",
  "predicted_stage_display": "Ripening",
  "confidence": 0.3643556833267212,
  "stage_belief": {
    "establishment": 0.08335071802139282,
    "tillering": 0.2347043752670288,
    "stem_booting": 0.12550415098667145,
    "reproductive": 0.17242158949375153,
    "grain_filling": 0.019663505256175995,
    "ripening": 0.3643556833267212
  }
}


## 23. Novelty experiments — ordinal/focal loss + fixed-split multi-seed (GPU)

Các thí nghiệm tăng novelty cho paper. Tất cả retrain ResNet-18 trên **cùng một split cố định** với `vision_final` (`OUTPUT_DIR/manifest.csv`) để so sánh trực tiếp với baseline CE:

- **ordinal** (Tier A#2): loss = CE + λ·(E[stage] − stage_thật)² → phạt mạnh lỗi cách xa giai đoạn, giảm MASD và lỗi non-adjacent.
- **focal** (Tier C#6): giảm trọng số mẫu dễ, giúp lớp hiếm (reproductive).
- **seed 7 / seed 123** (Tier C#9): cùng split cố định, chỉ đổi seed → tách phương sai khởi tạo khỏi phương sai split.

**Trước khi chạy:**
1. Đã `git push` code mới lên GitHub: `scripts/train_vision.py` (thêm `--loss`), `src/cropstate/losses.py`, `scripts/experiments/`.
2. Đã chạy lại **cell 2 (git pull)** để Colab có code mới.
3. Đã chạy **cell 19** (train `vision_final`) để có split cố định.

Colab có GPU nên **không** cần `CROPSTATE_FORCE_CPU`. Mỗi lần train ~vài phút trên GPU.

> **Cập nhật (quyết định phân tích):** ordinal và focal giờ chạy **3 seed {42, 7, 123}** giống CE
> để so sánh công bằng bằng mean±std. focal (C#6) bị **hạ khỏi danh sách đóng góp**
> (accuracy không tăng, ECE tăng vọt ~0.21) — chỉ giữ như ablation. Đóng góp vision chính là
> **ordinal (A#2)**: giảm MASD + tăng reproductive-recall, được kiểm định per-image (McNemar + bootstrap CI).

In [22]:
NOVELTY_DIR = f"{RESULTS_ROOT}/novelty"
FIXED_MANIFEST = f"{OUTPUT_DIR}/manifest.csv"  # split cố định từ baseline CE (cell 19)

import os
os.makedirs(NOVELTY_DIR, exist_ok=True)
assert os.path.exists(FIXED_MANIFEST), (
    f"Thiếu {FIXED_MANIFEST} — chạy cell 19 (train vision_final) trước để tạo split cố định."
)
print("Fixed split :", FIXED_MANIFEST)
print("Novelty dir :", NOVELTY_DIR)

Fixed split : /content/drive/MyDrive/CROPSTATE_RESULTS/vision_final/manifest.csv
Novelty dir : /content/drive/MyDrive/CROPSTATE_RESULTS/novelty


In [23]:
# Tier A#2 — ordinal loss (CE + λ·(E[stage] − stage_thật)²)
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 42 --loss ordinal \
  --output "{NOVELTY_DIR}/resnet18_ordinal"

[train] device=cuda | manifest 2900 rows
[epoch 1] phase=head_only val_acc=0.228 val_f1=0.093 val_loss=2.558
[epoch 2] phase=head_only val_acc=0.207 val_f1=0.118 val_loss=2.371
[epoch 3] phase=head_only val_acc=0.253 val_f1=0.218 val_loss=2.221
[epoch 4] phase=full_finetune val_acc=0.354 val_f1=0.434 val_loss=1.894
[epoch 5] phase=full_finetune val_acc=0.570 val_f1=0.543 val_loss=1.611
[epoch 6] phase=full_finetune val_acc=0.696 val_f1=0.646 val_loss=1.380
[epoch 7] phase=full_finetune val_acc=0.696 val_f1=0.631 val_loss=1.201
[epoch 8] phase=full_finetune val_acc=0.705 val_f1=0.643 val_loss=1.100
[epoch 9] phase=full_finetune val_acc=0.738 val_f1=0.676 val_loss=1.001
[epoch 10] phase=full_finetune val_acc=0.743 val_f1=0.698 val_loss=0.936
[epoch 11] phase=full_finetune val_acc=0.747 val_f1=0.720 val_loss=0.872
[epoch 12] phase=full_finetune val_acc=0.764 val_f1=0.711 val_loss=0.820
[epoch 13] phase=full_finetune val_acc=0.785 val_f1=0.770 val_loss=0.781
[epoch 14] phase=full_finetune 

In [24]:
# Tier A#2 — ordinal loss, seed 7 (multi-seed: tách phương sai khởi tạo)
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 7 --loss ordinal \
  --output "{NOVELTY_DIR}/resnet18_ordinal_seed7"

[train] device=cuda | manifest 2900 rows
[epoch 1] phase=head_only val_acc=0.194 val_f1=0.069 val_loss=2.551
[epoch 2] phase=head_only val_acc=0.207 val_f1=0.095 val_loss=2.360
[epoch 3] phase=head_only val_acc=0.266 val_f1=0.289 val_loss=2.185
[epoch 4] phase=full_finetune val_acc=0.464 val_f1=0.510 val_loss=1.849
[epoch 5] phase=full_finetune val_acc=0.641 val_f1=0.550 val_loss=1.570
[epoch 6] phase=full_finetune val_acc=0.675 val_f1=0.569 val_loss=1.357
[epoch 7] phase=full_finetune val_acc=0.688 val_f1=0.600 val_loss=1.214
[epoch 8] phase=full_finetune val_acc=0.722 val_f1=0.628 val_loss=1.100
[epoch 9] phase=full_finetune val_acc=0.713 val_f1=0.600 val_loss=0.998
[epoch 10] phase=full_finetune val_acc=0.726 val_f1=0.709 val_loss=0.947
[epoch 11] phase=full_finetune val_acc=0.717 val_f1=0.666 val_loss=0.888
[epoch 12] phase=full_finetune val_acc=0.734 val_f1=0.669 val_loss=0.856
[epoch 13] phase=full_finetune val_acc=0.759 val_f1=0.741 val_loss=0.788
[epoch 14] phase=full_finetune 

In [25]:
# Tier A#2 — ordinal loss, seed 123
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 123 --loss ordinal \
  --output "{NOVELTY_DIR}/resnet18_ordinal_seed123"

[train] device=cuda | manifest 2900 rows
[epoch 1] phase=head_only val_acc=0.190 val_f1=0.058 val_loss=2.567
[epoch 2] phase=head_only val_acc=0.194 val_f1=0.102 val_loss=2.377
[epoch 3] phase=head_only val_acc=0.312 val_f1=0.319 val_loss=2.206
[epoch 4] phase=full_finetune val_acc=0.443 val_f1=0.436 val_loss=1.865
[epoch 5] phase=full_finetune val_acc=0.549 val_f1=0.507 val_loss=1.566
[epoch 6] phase=full_finetune val_acc=0.662 val_f1=0.571 val_loss=1.337
[epoch 7] phase=full_finetune val_acc=0.700 val_f1=0.593 val_loss=1.181
[epoch 8] phase=full_finetune val_acc=0.722 val_f1=0.605 val_loss=1.075
[epoch 9] phase=full_finetune val_acc=0.705 val_f1=0.626 val_loss=0.990
[epoch 10] phase=full_finetune val_acc=0.747 val_f1=0.660 val_loss=0.909
[epoch 11] phase=full_finetune val_acc=0.738 val_f1=0.652 val_loss=0.874
[epoch 12] phase=full_finetune val_acc=0.768 val_f1=0.732 val_loss=0.817
[epoch 13] phase=full_finetune val_acc=0.768 val_f1=0.737 val_loss=0.791
[epoch 14] phase=full_finetune 

In [26]:
# Tier C#6 — focal loss (giúp lớp hiếm reproductive)
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 42 --loss focal \
  --output "{NOVELTY_DIR}/resnet18_focal"

[train] device=cuda | manifest 2900 rows
[epoch 1] phase=head_only val_acc=0.278 val_f1=0.244 val_loss=1.490
[epoch 2] phase=head_only val_acc=0.557 val_f1=0.504 val_loss=1.394
[epoch 3] phase=head_only val_acc=0.612 val_f1=0.569 val_loss=1.313
[epoch 4] phase=full_finetune val_acc=0.658 val_f1=0.627 val_loss=1.145
[epoch 5] phase=full_finetune val_acc=0.688 val_f1=0.636 val_loss=0.920
[epoch 6] phase=full_finetune val_acc=0.705 val_f1=0.664 val_loss=0.738
[epoch 7] phase=full_finetune val_acc=0.726 val_f1=0.693 val_loss=0.590
[epoch 8] phase=full_finetune val_acc=0.776 val_f1=0.764 val_loss=0.536
[epoch 9] phase=full_finetune val_acc=0.764 val_f1=0.750 val_loss=0.454
[epoch 10] phase=full_finetune val_acc=0.772 val_f1=0.765 val_loss=0.414
[epoch 11] phase=full_finetune val_acc=0.764 val_f1=0.744 val_loss=0.378
[epoch 12] phase=full_finetune val_acc=0.781 val_f1=0.748 val_loss=0.360
[epoch 13] phase=full_finetune val_acc=0.785 val_f1=0.763 val_loss=0.351
[epoch 14] phase=full_finetune 

In [27]:
# Tier C#6 — focal loss, seed 7 (multi-seed)
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 7 --loss focal \
  --output "{NOVELTY_DIR}/resnet18_focal_seed7"

[train] device=cuda | manifest 2900 rows
[epoch 1] phase=head_only val_acc=0.346 val_f1=0.289 val_loss=1.486
[epoch 2] phase=head_only val_acc=0.515 val_f1=0.464 val_loss=1.392
[epoch 3] phase=head_only val_acc=0.578 val_f1=0.548 val_loss=1.299
[epoch 4] phase=full_finetune val_acc=0.633 val_f1=0.593 val_loss=1.114
[epoch 5] phase=full_finetune val_acc=0.650 val_f1=0.547 val_loss=0.884
[epoch 6] phase=full_finetune val_acc=0.675 val_f1=0.615 val_loss=0.695
[epoch 7] phase=full_finetune val_acc=0.709 val_f1=0.695 val_loss=0.587
[epoch 8] phase=full_finetune val_acc=0.692 val_f1=0.623 val_loss=0.507
[epoch 9] phase=full_finetune val_acc=0.722 val_f1=0.660 val_loss=0.460
[epoch 10] phase=full_finetune val_acc=0.747 val_f1=0.742 val_loss=0.429
[epoch 11] phase=full_finetune val_acc=0.734 val_f1=0.734 val_loss=0.398
[epoch 12] phase=full_finetune val_acc=0.768 val_f1=0.741 val_loss=0.397
[epoch 13] phase=full_finetune val_acc=0.776 val_f1=0.755 val_loss=0.347
[epoch 14] phase=full_finetune 

In [28]:
# Tier C#6 — focal loss, seed 123
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 123 --loss focal \
  --output "{NOVELTY_DIR}/resnet18_focal_seed123"

[train] device=cuda | manifest 2900 rows
[epoch 1] phase=head_only val_acc=0.321 val_f1=0.287 val_loss=1.465
[epoch 2] phase=head_only val_acc=0.477 val_f1=0.394 val_loss=1.365
[epoch 3] phase=head_only val_acc=0.498 val_f1=0.440 val_loss=1.276
[epoch 4] phase=full_finetune val_acc=0.633 val_f1=0.535 val_loss=1.077
[epoch 5] phase=full_finetune val_acc=0.688 val_f1=0.639 val_loss=0.884
[epoch 6] phase=full_finetune val_acc=0.700 val_f1=0.576 val_loss=0.679
[epoch 7] phase=full_finetune val_acc=0.696 val_f1=0.599 val_loss=0.576
[epoch 8] phase=full_finetune val_acc=0.726 val_f1=0.664 val_loss=0.504
[epoch 9] phase=full_finetune val_acc=0.709 val_f1=0.654 val_loss=0.454
[epoch 10] phase=full_finetune val_acc=0.751 val_f1=0.737 val_loss=0.413
[epoch 11] phase=full_finetune val_acc=0.751 val_f1=0.720 val_loss=0.404
[epoch 12] phase=full_finetune val_acc=0.768 val_f1=0.745 val_loss=0.373
[epoch 13] phase=full_finetune val_acc=0.755 val_f1=0.742 val_loss=0.385
[epoch 14] phase=full_finetune 

In [29]:
# Tier C#9 — fixed-split, seed 7
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 7 --loss ce \
  --output "{NOVELTY_DIR}/resnet18_ce_seed7"

[train] device=cuda | manifest 2900 rows
[epoch 1] phase=head_only val_acc=0.502 val_f1=0.408 val_loss=1.676
[epoch 2] phase=head_only val_acc=0.549 val_f1=0.498 val_loss=1.594
[epoch 3] phase=head_only val_acc=0.578 val_f1=0.537 val_loss=1.519
[epoch 4] phase=full_finetune val_acc=0.641 val_f1=0.632 val_loss=1.365
[epoch 5] phase=full_finetune val_acc=0.637 val_f1=0.535 val_loss=1.196
[epoch 6] phase=full_finetune val_acc=0.667 val_f1=0.602 val_loss=1.050
[epoch 7] phase=full_finetune val_acc=0.688 val_f1=0.668 val_loss=0.927
[epoch 8] phase=full_finetune val_acc=0.688 val_f1=0.611 val_loss=0.844
[epoch 9] phase=full_finetune val_acc=0.713 val_f1=0.623 val_loss=0.774
[epoch 10] phase=full_finetune val_acc=0.734 val_f1=0.715 val_loss=0.737
[epoch 11] phase=full_finetune val_acc=0.730 val_f1=0.716 val_loss=0.696
[epoch 12] phase=full_finetune val_acc=0.759 val_f1=0.740 val_loss=0.673
[epoch 13] phase=full_finetune val_acc=0.776 val_f1=0.746 val_loss=0.622
[epoch 14] phase=full_finetune 

In [30]:
# Tier C#9 — fixed-split, seed 123
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml --model resnet18 --seed 123 --loss ce \
  --output "{NOVELTY_DIR}/resnet18_ce_seed123"

[train] device=cuda | manifest 2900 rows
[epoch 1] phase=head_only val_acc=0.481 val_f1=0.378 val_loss=1.669
[epoch 2] phase=head_only val_acc=0.603 val_f1=0.502 val_loss=1.596
[epoch 3] phase=head_only val_acc=0.633 val_f1=0.554 val_loss=1.519
[epoch 4] phase=full_finetune val_acc=0.637 val_f1=0.549 val_loss=1.361
[epoch 5] phase=full_finetune val_acc=0.692 val_f1=0.628 val_loss=1.200
[epoch 6] phase=full_finetune val_acc=0.688 val_f1=0.584 val_loss=1.042
[epoch 7] phase=full_finetune val_acc=0.713 val_f1=0.607 val_loss=0.919
[epoch 8] phase=full_finetune val_acc=0.726 val_f1=0.647 val_loss=0.832
[epoch 9] phase=full_finetune val_acc=0.722 val_f1=0.661 val_loss=0.766
[epoch 10] phase=full_finetune val_acc=0.747 val_f1=0.683 val_loss=0.704
[epoch 11] phase=full_finetune val_acc=0.785 val_f1=0.763 val_loss=0.682
[epoch 12] phase=full_finetune val_acc=0.797 val_f1=0.763 val_loss=0.640
[epoch 13] phase=full_finetune val_acc=0.781 val_f1=0.765 val_loss=0.635
[epoch 14] phase=full_finetune 

## 24. So sánh biến thể novelty với baseline CE

Đọc `test_metrics.json` của từng biến thể (cùng split cố định) và in bảng: accuracy, macro-F1, MASD, ECE, recall lớp reproductive. Kỳ vọng: **ordinal** giảm MASD và lỗi non-adjacent; **focal** tăng recall reproductive; **seed7/seed123** cho dải phương sai khởi tạo trên split cố định.

In [31]:
import json, os
import numpy as np, pandas as pd

# Mỗi họ loss = 3 seed trên CÙNG split cố định -> báo cáo mean +/- std,
# để phân biệt cải thiện thật với dao động do khởi tạo (seed).
families = {
    "CE":      [OUTPUT_DIR, f"{NOVELTY_DIR}/resnet18_ce_seed7", f"{NOVELTY_DIR}/resnet18_ce_seed123"],
    "ordinal": [f"{NOVELTY_DIR}/resnet18_ordinal", f"{NOVELTY_DIR}/resnet18_ordinal_seed7", f"{NOVELTY_DIR}/resnet18_ordinal_seed123"],
    "focal":   [f"{NOVELTY_DIR}/resnet18_focal", f"{NOVELTY_DIR}/resnet18_focal_seed7", f"{NOVELTY_DIR}/resnet18_focal_seed123"],
}
METRICS = ["accuracy", "macro_f1", "masd", "ece"]

def load_metric(d, key):
    p = os.path.join(d, "test_metrics.json")
    if not os.path.exists(p):
        return None
    m = json.load(open(p))
    if key == "reproductive_recall":
        return (m.get("per_class_recall") or {}).get("reproductive")
    return m.get(key)

rows = []
raw = {}
for fam, dirs in families.items():
    vals = {k: [load_metric(d, k) for d in dirs] for k in METRICS + ["reproductive_recall"]}
    vals = {k: [v for v in lst if v is not None] for k, lst in vals.items()}
    raw[fam] = vals
    n = max((len(v) for v in vals.values()), default=0)
    if n == 0:
        print("bỏ qua (chưa train seed nào):", fam); continue
    row = {"family": fam, "n_seeds": n}
    for k in METRICS + ["reproductive_recall"]:
        arr = np.array(vals[k], dtype=float)
        row[k] = f"{arr.mean():.3f} ± {arr.std(ddof=0):.3f}" if len(arr) else "—"
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(f"{NOVELTY_DIR}/retrain_comparison.csv", index=False)
print(df.to_string(index=False))
print("\nĐã lưu:", f"{NOVELTY_DIR}/retrain_comparison.csv")
print("\nGhi chú: nếu khoảng mean±std của ordinal và CE chồng lấn ở accuracy,")
print("KHÔNG được kết luận ordinal > CE về accuracy. Xem cell kiểm định per-image dưới đây")
print("để khẳng định cải thiện MASD/reproductive-recall nằm ngoài dao động seed.")

 family  n_seeds      accuracy      macro_f1          masd           ece reproductive_recall
     CE        3 0.807 ± 0.009 0.798 ± 0.019 0.223 ± 0.020 0.057 ± 0.015       0.861 ± 0.071
ordinal        3 0.802 ± 0.008 0.801 ± 0.006 0.234 ± 0.010 0.067 ± 0.014       0.917 ± 0.000
  focal        3 0.773 ± 0.006 0.764 ± 0.008 0.265 ± 0.005 0.173 ± 0.019       0.889 ± 0.039

Đã lưu: /content/drive/MyDrive/CROPSTATE_RESULTS/novelty/retrain_comparison.csv

Ghi chú: nếu khoảng mean±std của ordinal và CE chồng lấn ở accuracy,
KHÔNG được kết luận ordinal > CE về accuracy. Xem cell kiểm định per-image dưới đây
để khẳng định cải thiện MASD/reproductive-recall nằm ngoài dao động seed.


In [32]:
# Kiểm định per-image: ordinal vs CE, focal vs CE trên CÙNG test split (seed 42).
# Export logits cho 3 checkpoint rồi so per-image correctness và |pred-true| (thành phần MASD).
!PYTHONPATH=src python scripts/experiments/export_logits.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --checkpoint ce="{OUTPUT_DIR}/best_checkpoint.pt" \
  --checkpoint ordinal="{NOVELTY_DIR}/resnet18_ordinal/best_checkpoint.pt" \
  --checkpoint focal="{NOVELTY_DIR}/resnet18_focal/best_checkpoint.pt" \
  --output-dir "{NOVELTY_DIR}/logits_sig"

import sys, json, os
import numpy as np
sys.path.insert(0, "src")
from cropstate.statistics import paired_bootstrap_ci, paired_wilcoxon
from math import comb

SIG_DIR = f"{NOVELTY_DIR}/logits_sig"

def load(name):
    z = np.load(os.path.join(SIG_DIR, f"{name}.npz"), allow_pickle=True)
    logits, labels = z["test_logits"], z["test_labels"]
    ids = z["test_image_ids"]
    order = np.argsort(ids)  # căn cùng thứ tự ảnh giữa các checkpoint
    return logits[order].argmax(1), labels[order]

def mcnemar_exact(correct_a, correct_b):
    b = int(np.sum((correct_a == 1) & (correct_b == 0)))
    c = int(np.sum((correct_a == 0) & (correct_b == 1)))
    n = b + c
    if n == 0:
        return {"b": b, "c": c, "discordant": 0, "p_value": 1.0}
    k = min(b, c)
    p = min(1.0, 2.0 * sum(comb(n, i) for i in range(k + 1)) / (2 ** n))
    return {"b": b, "c": c, "discordant": n, "p_value": p}

pred_ce, y = load("ce")
results = {}
for name in ("ordinal", "focal"):
    pred_v, yv = load(name)
    assert np.array_equal(y, yv), "labels lệch — check split"
    corr_ce = (pred_ce == y).astype(int)
    corr_v  = (pred_v  == y).astype(int)
    dist_ce = np.abs(pred_ce - y).astype(float)   # per-image absolute stage distance
    dist_v  = np.abs(pred_v  - y).astype(float)
    acc_diff, acc_ci = paired_bootstrap_ci(corr_v, corr_ce)       # >0 => variant tốt hơn
    masd_diff, masd_ci = paired_bootstrap_ci(dist_v, dist_ce)     # <0 => variant tốt hơn
    results[f"{name}_vs_ce"] = {
        "mcnemar": mcnemar_exact(corr_v, corr_ce),
        "accuracy_diff": round(acc_diff, 4),
        "accuracy_diff_ci95": [round(x, 4) for x in acc_ci],
        "masd_diff": round(masd_diff, 4),
        "masd_diff_ci95": [round(x, 4) for x in masd_ci],
        "masd_wilcoxon": paired_wilcoxon(dist_v, dist_ce),
    }

os.makedirs(NOVELTY_DIR, exist_ok=True)
json.dump(results, open(f"{NOVELTY_DIR}/significance.json", "w"), indent=2)
print(json.dumps(results, indent=2))
print("\nĐọc: accuracy_diff_ci95 chứa 0 => khác biệt accuracy KHÔNG có ý nghĩa.")
print("masd_diff < 0 và CI không chứa 0 => cải thiện MASD có ý nghĩa (đây là claim của A#2).")

[export] ce: validation=237, test=263
[export] ordinal: validation=237, test=263
[export] focal: validation=237, test=263
{
  "ordinal_vs_ce": {
    "mcnemar": {
      "b": 14,
      "c": 9,
      "discordant": 23,
      "p_value": 0.4048728942871094
    },
    "accuracy_diff": 0.019,
    "accuracy_diff_ci95": [
      -0.0152,
      0.0532
    ],
    "masd_diff": -0.0304,
    "masd_diff_ci95": [
      -0.076,
      0.0152
    ],
    "masd_wilcoxon": {
      "statistic": 120.0,
      "p_value": 0.2057713746546469
    }
  },
  "focal_vs_ce": {
    "mcnemar": {
      "b": 11,
      "c": 15,
      "discordant": 26,
      "p_value": 0.557197093963623
    },
    "accuracy_diff": -0.0152,
    "accuracy_diff_ci95": [
      -0.0532,
      0.0228
    ],
    "masd_diff": 0.0076,
    "masd_diff_ci95": [
      -0.038,
      0.0494
    ],
    "masd_wilcoxon": {
      "statistic": 182.0,
      "p_value": 0.590013887163346
    }
  }
}

Đọc: accuracy_diff_ci95 chứa 0 => khác biệt accuracy KHÔNG có ý ng

In [ ]:
# Kiểm định RIÊNG cho claim thật của ordinal: reproductive-recall.
# ordinal không thắng accuracy/MASD có ý nghĩa (cell trên), nhưng ổn định recall
# lớp reproductive. Ở đây: (1) paired bootstrap + McNemar trên SUBSET ảnh test có
# nhãn thật = reproductive; (2) độ ổn định qua 3 seed (std).
import json, os, sys
import numpy as np
sys.path.insert(0, "src")
from cropstate.constants import STAGE_NAMES
from cropstate.statistics import paired_bootstrap_ci
from math import comb

REPRO = STAGE_NAMES.index("reproductive")
SIG_DIR = f"{NOVELTY_DIR}/logits_sig"

def _load_pred(name):
    z = np.load(os.path.join(SIG_DIR, f"{name}.npz"), allow_pickle=True)
    order = np.argsort(z["test_image_ids"])   # căn cùng thứ tự ảnh
    return z["test_logits"][order].argmax(1), z["test_labels"][order]

pred_ce, y = _load_pred("ce")
pred_or, y2 = _load_pred("ordinal")
assert np.array_equal(y, y2), "labels lệch giữa ce/ordinal"

mask = y == REPRO                                 # chỉ ảnh reproductive
ce_hit = (pred_ce[mask] == REPRO).astype(float)
or_hit = (pred_or[mask] == REPRO).astype(float)
n_repro = int(mask.sum())
diff, (lo, hi) = paired_bootstrap_ci(or_hit, ce_hit)   # >0 => ordinal recall cao hơn
b = int(((or_hit == 1) & (ce_hit == 0)).sum())   # ordinal cứu được, CE trượt
c = int(((or_hit == 0) & (ce_hit == 1)).sum())   # ordinal làm hỏng
n_disc = b + c
p_mcnemar = 1.0 if n_disc == 0 else min(
    1.0, 2.0 * sum(comb(n_disc, i) for i in range(min(b, c) + 1)) / (2 ** n_disc))

# Độ ổn định qua 3 seed (per_class_recall.reproductive từ test_metrics.json)
def _seed_recalls(dirs):
    out = []
    for d in dirs:
        p = os.path.join(d, "test_metrics.json")
        if os.path.exists(p):
            r = (json.load(open(p)).get("per_class_recall") or {}).get("reproductive")
            if r is not None:
                out.append(r)
    return np.array(out, dtype=float)

seed_dirs = {
    "CE":      [OUTPUT_DIR, f"{NOVELTY_DIR}/resnet18_ce_seed7", f"{NOVELTY_DIR}/resnet18_ce_seed123"],
    "ordinal": [f"{NOVELTY_DIR}/resnet18_ordinal", f"{NOVELTY_DIR}/resnet18_ordinal_seed7", f"{NOVELTY_DIR}/resnet18_ordinal_seed123"],
}
stability = {}
for fam, dirs in seed_dirs.items():
    arr = _seed_recalls(dirs)
    stability[fam] = {"n_seeds": len(arr),
                      "mean": round(float(arr.mean()), 4) if len(arr) else None,
                      "std": round(float(arr.std(ddof=0)), 4) if len(arr) else None}

repro_block = {
    "n_reproductive_test_images": n_repro,
    "recall_ce": round(float(ce_hit.mean()), 4),
    "recall_ordinal": round(float(or_hit.mean()), 4),
    "recall_diff_ord_minus_ce": round(diff, 4),
    "recall_diff_ci95": [round(lo, 4), round(hi, 4)],
    "mcnemar": {"b_ord_fixed": b, "c_ord_broke": c, "discordant": n_disc, "p_value": round(p_mcnemar, 4)},
    "multiseed_stability": stability,
}

sig_path = f"{NOVELTY_DIR}/significance.json"
sig = json.load(open(sig_path)) if os.path.exists(sig_path) else {}
sig["reproductive_recall"] = repro_block
json.dump(sig, open(sig_path, "w"), indent=2)
print(json.dumps(repro_block, indent=2))
print("\nĐọc kết quả:")
print(f"- Đúng hướng (ordinal {repro_block['recall_ordinal']:.3f} > CE {repro_block['recall_ce']:.3f}),")
print(f"  nhưng n={n_repro} ảnh reproductive là NHỎ: CI95 = {repro_block['recall_diff_ci95']}.")
print("- CI chạm/chứa 0 hoặc McNemar p>0.05 => single-seed CHƯA đủ lực thống kê.")
print("- Bằng chứng mạnh hơn là ĐỘ ỔN ĐỊNH: ordinal std nhỏ hơn CE nhiều qua các seed.")
print("  => phát biểu claim là 'ổn định recall', KHÔNG phải 'tăng recall có ý nghĩa'.")


In [ ]:
# Gia cố thống kê: GỘP 3 seed để tăng n cho kết luận reproductive.
# Dùng confusion_matrix có sẵn trong mỗi test_metrics.json (không cần export lại).
# Kết quả trung thực kỳ vọng: p vẫn > 0.05 -> khẳng định claim là "ỔN ĐỊNH", không phải "vượt trội".
import json, os
import numpy as np
from math import sqrt
from statistics import NormalDist
from cropstate.constants import STAGE_NAMES

REPRO = STAGE_NAMES.index("reproductive")
pool_dirs = {
    "CE":      [OUTPUT_DIR, f"{NOVELTY_DIR}/resnet18_ce_seed7", f"{NOVELTY_DIR}/resnet18_ce_seed123"],
    "ordinal": [f"{NOVELTY_DIR}/resnet18_ordinal", f"{NOVELTY_DIR}/resnet18_ordinal_seed7", f"{NOVELTY_DIR}/resnet18_ordinal_seed123"],
}

def pool(dirs):
    correct = total = 0
    per_seed = []
    for d in dirs:
        cm = np.array(json.load(open(os.path.join(d, "test_metrics.json")))["confusion_matrix"])
        row = cm[REPRO]
        correct += int(row[REPRO]); total += int(row.sum())
        per_seed.append(round(float(row[REPRO] / row.sum()), 4))
    return correct, total, per_seed

c1, n1, ps_or = pool(pool_dirs["ordinal"])
c0, n0, ps_ce = pool(pool_dirs["CE"])
p1, p0 = c1 / n1, c0 / n0
p = (c1 + c0) / (n1 + n0)
z = (p1 - p0) / sqrt(p * (1 - p) * (1 / n1 + 1 / n0))
pval = 2 * (1 - NormalDist().cdf(abs(z)))

pooled_block = {
    "n_pooled_per_family": n1,
    "ce": {"correct": c0, "total": n0, "recall": round(p0, 4), "per_seed": ps_ce, "std": round(float(np.std(ps_ce)), 4)},
    "ordinal": {"correct": c1, "total": n1, "recall": round(p1, 4), "per_seed": ps_or, "std": round(float(np.std(ps_or)), 4)},
    "two_proportion_z": round(z, 3), "p_value": round(pval, 3),
}

sig_path = f"{NOVELTY_DIR}/significance.json"
sig = json.load(open(sig_path)) if os.path.exists(sig_path) else {}
sig.setdefault("reproductive_recall", {})["pooled_multiseed"] = pooled_block
json.dump(sig, open(sig_path, "w"), indent=2)
print(json.dumps(pooled_block, indent=2))
print(f"\nĐọc: gộp n={n1} (3 seed x {n1//3} ảnh). ordinal {p1:.3f} vs CE {p0:.3f}, p={pval:.3f}.")
print("- p>0.05 => chênh lệch TRUNG BÌNH vẫn KHÔNG có ý nghĩa dù đã gộp.")
print(f"- NHƯNG ordinal std={pooled_block['ordinal']['std']:.3f} vs CE std={pooled_block['ce']['std']:.3f}")
print("  => bằng chứng thật là ĐỘ ỔN ĐỊNH (ordinal cho cùng một kết quả mọi seed).")
print("  => paper phát biểu 'stable recall', và ghi rõ ở Limitations là chưa đủ lực để claim 'higher'.")


## 25. (Tuỳ chọn) Phân tích post-hoc trên Colab → lưu vào Drive

> **A#3 (temperature scaling) — reframe thành SELECTIVE PREDICTION.** Trên test, ECE gần như
> không đổi (0.072 → 0.070), nên KHÔNG trình bày như "cải thiện calibration". Thay vào đó khai thác
> đường **risk–coverage / abstention** (cell dưới): bỏ qua ~37% ca ít tự tin nhất thì selective-risk
> giảm còn ~0.10 (từ 0.205), AURC≈0.125 — một kết quả triển khai thực, có giá trị. Đây mới là đóng góp
> giữ lại từ A#3. **B#5 (temporal fusion)** chỉ là _controlled demo_ trên quỹ đạo mô phỏng
> (không có time-series thật) → để ở future-work, không phải bằng chứng thực nghiệm.

Export logits từ các checkpoint có sẵn rồi chạy calibration (temperature scaling, Tier A#3),
temporal fusion (Tier B#5) và leakage audit (Tier A#1). Cần checkpoint `vision_final`
(và nếu có `vision_small_cnn`, `vision_efficientnet_b0`) trong `RESULTS_ROOT`.
Kết quả JSON lưu vào `RESULTS_ROOT/novelty` để kéo về.

In [33]:
CKPTS = {
    "resnet18": f"{OUTPUT_DIR}/best_checkpoint.pt",
    "small_cnn": f"{RESULTS_ROOT}/vision_small_cnn/best_checkpoint.pt",
    "efficientnet_b0": f"{RESULTS_ROOT}/vision_efficientnet_b0/best_checkpoint.pt",
}
import os
ck = " ".join(f'--checkpoint {n}={p}' for n, p in CKPTS.items() if os.path.exists(p))
print("Checkpoints dùng được:", ck or "(chỉ resnet18)")

!PYTHONPATH=src python scripts/experiments/export_logits.py {ck} \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" \
  --output-dir "{NOVELTY_DIR}/logits"

# Calibration (temperature scaling) — torch-free, cắt ECE
!PYTHONPATH=src python scripts/experiments/calibration.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" \
  --output "{NOVELTY_DIR}/calibration.json"

# Temporal phenology fusion (Tier B#5)
!PYTHONPATH=src python scripts/experiments/temporal_fusion.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" --model resnet18 \
  --output "{NOVELTY_DIR}/temporal_fusion.json"

# Near-duplicate / leakage audit (Tier A#1)
!PYTHONPATH=src python scripts/experiments/leakage_audit.py \
  --manifest "{FIXED_MANIFEST}" --data-root "{DATA_ROOT}" --hamming-threshold 10 \
  --output "{NOVELTY_DIR}/leakage_audit.json"

print("Xong. Kết quả trong", NOVELTY_DIR)

Checkpoints dùng được: --checkpoint resnet18=/content/drive/MyDrive/CROPSTATE_RESULTS/vision_final/best_checkpoint.pt
[export] resnet18: validation=237, test=263
[resnet18] T=0.865  ECE 0.072 -> 0.070   Brier 0.300 -> 0.298   NLL 0.537 -> 0.543
{
  "single_frame_argmax": {
    "accuracy": 0.7946768060836502,
    "masd": 0.2509505703422053,
    "non_adjacent_errors": 7,
    "adjacent_errors": 47
  },
  "temporal_fused_mean_over_20_orderings": {
    "accuracy": 0.8199619771863119,
    "masd": 0.1855513307984791,
    "non_adjacent_errors": 1.3,
    "adjacent_errors": 46.05
  },
  "non_adjacent_error_reduction": 5.7,
  "masd_reduction": 0.0653992395437262
}
{
  "threshold_sweep": [
    {
      "hamming_threshold": 4,
      "near_duplicate_edges": 39,
      "multi_image_clusters": 36,
      "largest_cluster_size": 3,
      "clusters_straddling_splits": 0,
      "cross_split_pairs": 0,
      "test_or_val_leak_pairs": 0
    },
    {
      "hamming_threshold": 6,
      "near_duplicate_edges": 

In [ ]:
# A#3 reframe: từ "calibration" (ECE hầu như không đổi) sang SELECTIVE PREDICTION.
# Dùng abstention curve (đã có trong calibration.json) để dựng đường risk–coverage,
# tính AURC, và xuất bảng + hình cho paper.
import json, os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cal = json.load(open(f"{NOVELTY_DIR}/calibration.json"))["resnet18"]
curve = cal["abstention_curve_after"]
thr = np.array([p["threshold"] for p in curve])
cov = np.array([p["coverage"] for p in curve])
acc = np.array([p["selective_accuracy"] for p in curve])
risk = 1.0 - acc

order = np.argsort(cov)
c_sorted, r_sorted = cov[order], risk[order]
_trap = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
aurc = float(_trap(r_sorted, c_sorted) / (c_sorted.max() - c_sorted.min()))
full_risk = float(risk[cov.argmax()])            # risk ở coverage cao nhất (~1.0)

# Bảng CSV
import csv
sp_csv = f"{NOVELTY_DIR}/selective_prediction.csv"
with open(sp_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["threshold", "coverage", "selective_accuracy", "selective_risk"])
    for t, cv, ac in zip(thr, cov, acc):
        w.writerow([f"{t:.2f}", f"{cv:.4f}", f"{ac:.4f}", f"{1 - ac:.4f}"])

# Điểm vận hành đáng chú ý
def at_cov(target):
    i = int(np.argmin(np.abs(cov - target)))
    return cov[i], acc[i], thr[i]
summary = {
    "full_coverage_risk": round(full_risk, 4),
    "aurc_normalized": round(aurc, 4),
    "operating_points": {
        f"cov~{tg:.2f}": {"coverage": round(float(a), 4), "selective_accuracy": round(float(b), 4),
                          "threshold": round(float(t), 2)}
        for tg in (1.0, 0.75, 0.50) for a, b, t in [at_cov(tg)]
    },
}
json.dump(summary, open(f"{NOVELTY_DIR}/selective_prediction.json", "w"), indent=2)

# Hình risk–coverage
fig, ax = plt.subplots(figsize=(5, 3.4), dpi=140)
ax.plot(cov[order], risk[order], "-o", ms=4, lw=1.6, color="#1f77b4", label="ResNet-18 (softmax conf.)")
ax.axhline(full_risk, ls="--", lw=1, color="grey", label=f"full-coverage risk = {full_risk:.3f}")
ax.set_xlabel("Coverage (tỉ lệ ảnh được dự đoán)")
ax.set_ylabel("Selective risk (1 − accuracy)")
ax.set_title(f"Risk–coverage (AURC={aurc:.3f})")
ax.invert_xaxis()
ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
sp_png = f"{NOVELTY_DIR}/selective_prediction.png"
fig.savefig(sp_png, bbox_inches="tight"); plt.close(fig)

print(json.dumps(summary, indent=2))
print("\nĐã lưu:", sp_csv, "|", sp_png, "|", f"{NOVELTY_DIR}/selective_prediction.json")
print(f"Thông điệp: bỏ qua ~{(1 - at_cov(0.75)[0]) * 100:.0f}% ca ít tự tin nhất thì risk giảm còn")
print(f"{1 - at_cov(0.75)[1]:.3f} (từ {full_risk:.3f}) — đây là đóng góp cứu lại từ Tier A#3,")
print("KHÔNG trình bày dưới danh nghĩa 'giảm ECE' (ECE gần như không đổi).")


## 12. Retrieval đóng vòng từ belief thật (robust)

`evaluate_retrieval` ở mục trước chỉ có 2 kịch bản minh hoạ (suy biến về 0). Ở đây sinh kịch bản
từ **belief 6 lớp thật của mọi ảnh test** (logits vừa export), gán relevance tự động theo
`stage_compatibility` của từng chunk IRRI (không cần chuyên gia gán tay). Chỉ giữ cặp (topic, stage)
mà corpus trả lời được (≥3 relevant & ≥3 incompatible) → **420 kịch bản**, đủ lực thống kê.

> **Kết luận (đã kiểm định trên 420 kịch bản, paired bootstrap — xem cell dưới):**
> `P_adaptive_soft` (đề xuất) **thắng có ý nghĩa** `B0_ungated` và `B3_fixed_soft` trên **mọi** metric
> (nDCG, recall, và sIRR — an-toàn-stage). So với baseline mạnh `B1_query_expansion`, nDCG/recall
> **HOÀ** (CI chứa 0) nhưng `P_adaptive_soft` có **sIRR thấp hơn ~0.12** (≈3× ít chunk sai-stage hơn)
> — đúng thuộc tính an toàn mà CROPSTATE nhắm tới. → Trình bày `P_adaptive_soft` là phương pháp chính
> với luận điểm **"ngang chất lượng xếp hạng, an-toàn-stage vượt trội"**; B1 mạnh precision thô nhưng
> kéo nhiều chunk sai-stage (sIRR cao nhất). `B2_hard_filter` phá sIRR về ~0.002 nhưng cứng nhắc → chỉ nêu như cận trên của hard-gating.

In [34]:
# Sinh kịch bản từ belief thật + corpus IRRI, rồi đánh giá retrieval (đóng vòng).
!PYTHONPATH=src python scripts/experiments/build_belief_scenarios.py \
  --logits-index "{NOVELTY_DIR}/logits/index.json" --model resnet18 \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --calibration "{NOVELTY_DIR}/calibration.json" \
  --output "{NOVELTY_DIR}/belief_scenarios.csv"

!PYTHONPATH=src python scripts/evaluate_retrieval.py \
  --corpus "{KB_NONRESTRICTED_CORPUS}" \
  --scenarios "{NOVELTY_DIR}/belief_scenarios.csv" \
  --mode research \
  --output "{RESULTS_ROOT}/retrieval/retrieval_evaluation_belief.json"

[scenarios] wrote 420 scenarios over 5 topics, 263 test images, temperature=0.865
Loading weights: 100% 199/199 [00:00<00:00, 5246.04it/s]
{
  "B0_ungated": {
    "precision_at_k": 0.22476190476190477,
    "recall_at_k": 0.11832548403976974,
    "ndcg_at_k": 0.16509145548789161,
    "sirr_at_k": 0.1357142857142857
  },
  "B1_query_expansion": {
    "precision_at_k": 0.37142857142857144,
    "recall_at_k": 0.1930715754587935,
    "ndcg_at_k": 0.33632025031078644,
    "sirr_at_k": 0.15857142857142856
  },
  "B2_hard_filter": {
    "precision_at_k": 0.2666666666666667,
    "recall_at_k": 0.1808484581416912,
    "ndcg_at_k": 0.25890293919780966,
    "sirr_at_k": 0.002380952380952381
  },
  "B3_fixed_soft": {
    "precision_at_k": 0.2614285714285714,
    "recall_at_k": 0.16367099066347185,
    "ndcg_at_k": 0.2666185166753228,
    "sirr_at_k": 0.07238095238095238
  },
  "P_adaptive_soft": {
    "precision_at_k": 0.30095238095238097,
    "recall_at_k": 0.19868978765595305,
    "ndcg_at_k": 0.

In [ ]:
# Kiểm định retrieval trên 420 kịch bản belief: paired bootstrap CI giữa
# P_adaptive_soft và các baseline, theo từng metric. Đây là bằng chứng ở QUY MÔ,
# thay cho bản 2-kịch-bản (suy biến). Lưu retrieval_significance.json.
import json, os, sys
import numpy as np
from collections import defaultdict
sys.path.insert(0, "src")
from cropstate.statistics import paired_bootstrap_ci

ev = json.load(open(f"{RESULTS_ROOT}/retrieval/retrieval_evaluation_belief.json"))
rows = ev["rows"]
by = defaultdict(dict)                    # scenario -> method -> row
for r in rows:
    by[r["scenario_id"]][r["method"]] = r
scenarios = sorted(by.keys())

def vec(method, metric):
    return np.array([by[s][method][metric] for s in scenarios if method in by[s]])

PROP = "P_adaptive_soft"
BASELINES = ["B0_ungated", "B1_query_expansion", "B3_fixed_soft"]
# sIRR: THẤP hơn = tốt hơn (tỉ lệ chunk sai-stage). Các metric còn lại: cao = tốt.
METRICS = ["ndcg_at_k", "recall_at_k", "precision_at_k", "sirr_at_k"]
LOWER_IS_BETTER = {"sirr_at_k"}

out = {"n_scenarios": len(scenarios), "proposed": PROP, "comparisons": {}}
for metric in METRICS:
    p = vec(PROP, metric)
    block = {}
    for base in BASELINES:
        o = vec(base, metric)
        d, (lo, hi) = paired_bootstrap_ci(p, o)   # proposed - baseline
        favors = None
        if lo > 0 or hi < 0:
            better = (d > 0) != (metric in LOWER_IS_BETTER)   # đảo dấu cho sIRR
            favors = PROP if better else base
        block[base] = {"diff_prop_minus_base": round(d, 4),
                       "ci95": [round(lo, 4), round(hi, 4)],
                       "significant": bool(lo > 0 or hi < 0),
                       "favors": favors}
    out["comparisons"][metric] = block

sig_path = f"{RESULTS_ROOT}/retrieval/retrieval_significance.json"
json.dump(out, open(sig_path, "w"), indent=2)
print(json.dumps(out, indent=2))
print("\nDiễn giải (n=420):")
print("- vs B3_fixed_soft: nếu significant & favors P_adaptive trên mọi metric => adaptive > fixed.")
print("- vs B1_query_expansion: nDCG/recall có thể HOÀ (CI chứa 0), nhưng sIRR của P_adaptive")
print("  thấp hơn rõ rệt => cùng chất lượng xếp hạng nhưng AN TOÀN-stage hơn nhiều.")
print("- vs B0_ungated: P_adaptive thắng => gating có tác dụng.")
print("Đã lưu:", sig_path)


In [ ]:
# Hình cho paper: đánh đổi CHẤT LƯỢNG (nDCG, cao=tốt) vs AN-TOÀN-STAGE (sIRR, thấp=tốt).
# Trực quan hoá luận điểm: P_adaptive nằm ở góc lý tưởng (gần oracle), B1 chất lượng cao nhưng kém an toàn.
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ev = json.load(open(f"{RESULTS_ROOT}/retrieval/retrieval_evaluation_belief.json"))
s = ev["summary"]
methods = ["B0_ungated", "B1_query_expansion", "B2_hard_filter", "B3_fixed_soft", "P_adaptive_soft", "oracle_reference"]
labels = {"B0_ungated": "B0 no-gating", "B1_query_expansion": "B1 query-exp", "B2_hard_filter": "B2 hard-filter",
          "B3_fixed_soft": "B3 fixed-soft", "P_adaptive_soft": "P adaptive (ours)", "oracle_reference": "oracle"}

fig, ax = plt.subplots(figsize=(6, 4.2), dpi=140)
for m in methods:
    x, y = s[m]["ndcg_at_k"], s[m]["sirr_at_k"]
    ours, orc = m == "P_adaptive_soft", m == "oracle_reference"
    col = "#d62728" if ours else ("#2ca02c" if orc else "#1f77b4")
    ax.scatter(x, y, s=170 if ours else 90, c=col, marker="*" if ours else ("D" if orc else "o"),
               edgecolors="black", linewidths=0.8, zorder=3)
    ax.annotate(labels[m], (x, y), xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("nDCG@5  (chất lượng xếp hạng — cao = tốt →)")
ax.set_ylabel("sIRR@5  (tỉ lệ chunk sai-stage — thấp = an toàn ↓)")
ax.invert_yaxis()
ax.set_title(f"Đánh đổi chất lượng vs an-toàn-stage (n={ev['scenario_count']} kịch bản)")
ax.grid(alpha=0.3)
fig.tight_layout()
out_png = f"{RESULTS_ROOT}/retrieval/retrieval_tradeoff.png"
fig.savefig(out_png, bbox_inches="tight"); plt.close(fig)
print("Đã lưu:", out_png)
print("Đọc hình: P_adaptive (sao đỏ) ở góc lý tưởng gần oracle; B1 nDCG cao nhất nhưng sIRR tệ nhất.")


## 26. Claim đã kiểm định & Limitations (paper-ready)

Bảng dưới là phát biểu **được số liệu chống lưng** cho từng đóng góp, sau khi multi-seed + kiểm định.
Cột "Phát biểu an toàn" là câu nên viết trong paper; cột "KHÔNG viết" là bẫy đã tránh.

| Tier | Bằng chứng | Phát biểu an toàn (dùng được) | KHÔNG viết |
|---|---|---|---|
| **A#1 leakage audit** | 0 cặp leak test/val ở Hamming≤10 (`leakage_audit.json`) | "Split leak-free tới ngưỡng near-duplicate hợp lý" | — (đóng góp mạnh nhất, để headline) |
| **A#2 ordinal** | recall reproductive 0.917±**0.000** vs CE 0.861±**0.071**; accuracy/MASD hoà (CI chứa 0) | "Ordinal **ổn định hóa** recall lớp reproductive qua các seed, không đánh đổi accuracy" | "Ordinal tăng accuracy/MASD có ý nghĩa" |
| **A#3 → selective pred.** | AURC=0.125; risk 0.205→~0.10 khi coverage 63% (`selective_prediction.*`) | "Selective prediction cắt nửa risk khi bỏ ~37% ca ít tự tin" | "Temperature scaling cải thiện ECE" (ECE ~ không đổi) |
| **B#5 temporal fusion** | `note: no real time-series`; MASD 0.251→0.186 trên quỹ đạo mô phỏng | (future-work) | "Temporal fusion cải thiện trên dữ liệu thật" |
| **C#6 focal** | ECE 0.173 (gấp 3× CE), accuracy thấp nhất | (ablation tiêu cực) | "Focal giúp lớp hiếm" |
| **Retrieval P_adaptive** | n=420: thắng B0/B3 mọi metric; ngang B1 (nDCG/recall) nhưng sIRR −0.12 (`retrieval_significance.json`) | "Ngang chất lượng xếp hạng baseline mạnh nhất, nhưng an-toàn-stage vượt trội (~3× ít chunk sai-stage)" | "P_adaptive thắng mọi baseline mọi metric" |

**Đoạn Limitations cho paper (bê thẳng vào):**

> Reproductive stage has n=24 test images; per-image significance is underpowered
> (McNemar p=0.50), so we frame ordinal's benefit as recall **stability across seeds**
> (σ=0 vs 0.071) rather than a significant single-run gain. Retrieval relevance is
> auto-derived from stage-compatibility, not expert-annotated. Temporal fusion uses
> simulated trajectories.

## Ghi chú

- GitHub chỉ lưu code và metadata nhỏ.
- Google Drive lưu dataset, knowledge base, checkpoint và kết quả training.
- Không commit các file `.pt` lên GitHub.